# 02 — Análise Exploratória de Dados (EDA)

Este notebook investiga os dados de voos com estatísticas descritivas e visualizações para extrair insights sobre padrões de atrasos.

In [1]:
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import pandas as pd
import matplotlib.pyplot as plt

from voos.eda import (
    estatisticas_descritivas,
    distribuicao_atrasos,
    atrasos_por_companhia,
    atrasos_por_aeroporto,
    atrasos_por_dia_semana,
    atrasos_por_mes,
    correlacao_variaveis,
)

# Carregar dados limpos
CAMINHO_PARQUET = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "voos_limpos.parquet")
df = pd.read_parquet(CAMINHO_PARQUET)
print(f"Dados carregados: {len(df):,} linhas")

Dados carregados: 5,819,079 linhas


## 1. Estatísticas descritivas

Resumo estatístico das colunas de atraso — média, mediana, desvio padrão, quartis. A mediana é uma métrica mais robusta que a média para dados com outliers (atrasos extremos).

In [2]:
stats = estatisticas_descritivas(df)
for chave, valores in stats.items():
    print(f"\n{chave.upper()}:")
    if isinstance(valores, dict) and "DEPARTURE_DELAY" in valores:
        for col, val in valores.items():
            print(f"  {col}: {val:.2f}" if isinstance(val, (int, float)) else f"  {col}: {val}")
    else:
        print(f"  {valores}")


MEDIA:
  DEPARTURE_DELAY: 9.37
  ARRIVAL_DELAY: 4.41

MEDIANA:
  DEPARTURE_DELAY: -2.00
  ARRIVAL_DELAY: -5.00

DESVIO_PADRAO:
  DEPARTURE_DELAY: 37.08
  ARRIVAL_DELAY: 39.27

MIN:
  DEPARTURE_DELAY: -82.00
  ARRIVAL_DELAY: -87.00

MAX:
  DEPARTURE_DELAY: 1988.00
  ARRIVAL_DELAY: 1971.00

QUARTIS:
  DEPARTURE_DELAY: {'25%': -5.0, '50%': -2.0, '75%': 7.0}
  ARRIVAL_DELAY: {'25%': -13.0, '50%': -5.0, '75%': 8.0}


## 2. Distribuição dos atrasos

A maioria dos voos parte próximo do horário previsto. A distribuição é fortemente assimétrica à direita — poucos voos concentram atrasos muito longos. O limiar de 15 minutos (padrão FAA) separa voos "atrasados" de "no horário".

In [3]:
fig = distribuicao_atrasos(df)
fig.show()

/var/folders/sr/gspz6kpn1ml0w6pnmmzzprwr0000gn/T/ipykernel_42179/314227219.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()


## 3. Atrasos por companhia aérea

Companhias aéreas com operações maiores em hubs congestionados tendem a ter atrasos médios mais altos. Diferenças operacionais e de frota também impactam.

In [4]:
fig = atrasos_por_companhia(df)
fig.show()

/Users/nsx001191/Documents/fiap/fiap-tcf-3/src/voos/eda.py:66: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  media = df_valido.groupby("AIRLINE")["DEPARTURE_DELAY"].mean().sort_values(ascending=False)
/var/folders/sr/gspz6kpn1ml0w6pnmmzzprwr0000gn/T/ipykernel_42179/1788003087.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()


## 4. Aeroportos mais críticos

Aeroportos menores ou regionais podem ter atrasos médios maiores, mas com volume menor. Os grandes hubs (ATL, ORD, LAX) acumulam mais volume absoluto de atrasos.

In [5]:
fig = atrasos_por_aeroporto(df, top_n=20)
fig.show()

/Users/nsx001191/Documents/fiap/fiap-tcf-3/src/voos/eda.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_valido.groupby("ORIGIN_AIRPORT")["DEPARTURE_DELAY"]
/var/folders/sr/gspz6kpn1ml0w6pnmmzzprwr0000gn/T/ipykernel_42179/4209083393.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()


## 5. Padrões temporais

### 5.1 Atrasos por dia da semana
Dias úteis no meio da semana (terça a quinta) costumam ter menor volume de atrasos, enquanto sextas e domingos são mais críticos devido ao fluxo de viajantes.

In [6]:
fig = atrasos_por_dia_semana(df)
fig.show()

/var/folders/sr/gspz6kpn1ml0w6pnmmzzprwr0000gn/T/ipykernel_42179/3709121142.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()


### 5.2 Atrasos por mês
Meses de verão (junho-agosto) e período de festas (dezembro) costumam apresentar mais atrasos devido ao aumento da demanda e condições meteorológicas adversas.

In [7]:
fig = atrasos_por_mes(df)
fig.show()

/var/folders/sr/gspz6kpn1ml0w6pnmmzzprwr0000gn/T/ipykernel_42179/2007922954.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()


## 6. Correlações

O heatmap de correlação revela relações entre variáveis numéricas. Esperamos forte correlação entre `DEPARTURE_DELAY` e `ARRIVAL_DELAY`, e entre as colunas de tempo (SCHEDULED_TIME, ELAPSED_TIME, AIR_TIME, DISTANCE).

In [8]:
fig = correlacao_variaveis(df)
fig.show()

/var/folders/sr/gspz6kpn1ml0w6pnmmzzprwr0000gn/T/ipykernel_42179/4119631924.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()
